# **SUPPORT VECTOR MACHINE**

In [1]:
from utils import *

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, confusion_matrix

In [2]:
# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [3]:
# -----------------------
# Model building function
# -----------------------

def build_model(C=10.0, penalty="l1", loss="squared_hinge", dual=False):
    """
    Builds a scikit-learn Pipeline with TF-IDF vectorization and SVM classifier.

    Args:
        C (float): Inverse of regularization strength for SVM.
        penalty (str): Regularization type for SVM.
        loss (str): Loss function to use for SVM.
        dual (bool): Whether to solve the dual optimization problem.

    Returns:
        Pipeline: A scikit-learn Pipeline object (TF-IDF + SVM).
    """
    
    return Pipeline([
        ('tfidf', TfidfVectorizer(  # TD-IDF vectorization inside pipeline
            max_features=5000,      # limit to top 5000 features
            ngram_range=(1, 2),     # unigrams + bigrams
            stop_words="english"    # remove English stop words
        )),
        ('clf', LinearSVC(         # Linear SVM classifier
            C=C,
            penalty=penalty,
            loss=loss,
            dual=dual,
            max_iter=2000,
            random_state=42
        ))
    ])

## VERSION 1: Dataset (Simple)

In [4]:
dataset_df = data_loading()

for name, df in dataset_df.items():
    print(f"Dataset: {name}, Number of samples: {len(df)}")

Dataset: Celebrity, Number of samples: 500
Dataset: CIDII, Number of samples: 722
Dataset: FaKES, Number of samples: 842
Dataset: FakeVsSatire, Number of samples: 486
Dataset: Horne, Number of samples: 326
Dataset: Infodemic, Number of samples: 10559
Dataset: ISOT, Number of samples: 44271
Dataset: Kaggle_clement, Number of samples: 39105
Dataset: Kaggle_meg, Number of samples: 12845
Dataset: LIAR_PLUS, Number of samples: 12784
Dataset: Politifact, Number of samples: 504
Dataset: Unipi_NDF, Number of samples: 554


In [5]:
# --------------------------------
# Fine-tuning on multiple datasets
# --------------------------------

datasets = {name: split_dataset(df) for name, df in dataset_df.items()} # split all datasets in train/val/test
model = build_model() # initialize model

results = {}

# sequential training
for i, (name, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on {name} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val])) # fine-tune on train + val

    y_pred = model.predict(X_test)
    print(f"Classification Report after {name}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after {name}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after {name}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all datasets
    print("\n--- Evaluation on all datasets ---")
    results[name] = {}
    for test_name, test_data in datasets.items(): # for each dataset
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        f1 = f1_score(y_te, preds, average="weighted")
        results[name][test_name] = f1
        print(f"Evaluation on {test_name}: Weighted F1 = {f1:.4f}")



=== Phase 1: Training/Fine-tuning on Celebrity ===
Classification Report after Celebrity:
              precision    recall  f1-score   support

           0       0.58      0.68      0.62        50
           1       0.61      0.50      0.55        50

    accuracy                           0.59       100
   macro avg       0.59      0.59      0.59       100
weighted avg       0.59      0.59      0.59       100

Confusion Matrix after Celebrity:
[[34 16]
 [25 25]]

Weighted F1-score after Celebrity: 0.5866518802298619

--- Evaluation on all datasets ---
Evaluation on Celebrity: Weighted F1 = 0.5867
Evaluation on CIDII: Weighted F1 = 0.5747
Evaluation on FaKES: Weighted F1 = 0.4011
Evaluation on FakeVsSatire: Weighted F1 = 0.4506
Evaluation on Horne: Weighted F1 = 0.6382
Evaluation on Infodemic: Weighted F1 = 0.4439
Evaluation on ISOT: Weighted F1 = 0.4919
Evaluation on Kaggle_clement: Weighted F1 = 0.5220
Evaluation on Kaggle_meg: Weighted F1 = 0.8742
Evaluation on LIAR_PLUS: Weighte

In [6]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for name, res in results.items():
    print(f"\nResults after training on {name}:")
    for test_name, f1 in res.items():
        print(f"  Test on {test_name}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on Celebrity:
  Test on Celebrity: Weighted F1 = 0.5867
  Test on CIDII: Weighted F1 = 0.5747
  Test on FaKES: Weighted F1 = 0.4011
  Test on FakeVsSatire: Weighted F1 = 0.4506
  Test on Horne: Weighted F1 = 0.6382
  Test on Infodemic: Weighted F1 = 0.4439
  Test on ISOT: Weighted F1 = 0.4919
  Test on Kaggle_clement: Weighted F1 = 0.5220
  Test on Kaggle_meg: Weighted F1 = 0.8742
  Test on LIAR_PLUS: Weighted F1 = 0.4593
  Test on Politifact: Weighted F1 = 0.5542
  Test on Unipi_NDF: Weighted F1 = 0.4245

Results after training on CIDII:
  Test on Celebrity: Weighted F1 = 0.3967
  Test on CIDII: Weighted F1 = 0.8663
  Test on FaKES: Weighted F1 = 0.3803
  Test on FakeVsSatire: Weighted F1 = 0.4202
  Test on Horne: Weighted F1 = 0.6812
  Test on Infodemic: Weighted F1 = 0.4553
  Test on ISOT: Weighted F1 = 0.4186
  Test on Kaggle_clement: Weighted F1 = 0.4727
  Test on Kaggle_meg: Weighted F1 = 0.9002
  Test on LIAR_PLUS: Weighted F1 = 0

## VERSION 2: Dataset by Topic

In [7]:
dataset_df = data_by_topic()

for topic, df in dataset_df.items():
    print(f"Topic: {topic}, Number of samples: {len(df)}")

Topic: politics, Number of samples: 97476
Topic: general, Number of samples: 12845
Topic: covid, Number of samples: 10559
Topic: syria, Number of samples: 842
Topic: islam, Number of samples: 722
Topic: notredame, Number of samples: 554
Topic: gossip, Number of samples: 500


In [8]:
# -------------------------------
# Fine-tuning on Dataset by Topic
# -------------------------------

datasets = {topic: split_dataset(df) for topic, df in dataset_df.items()} # split all datasets in train/val/test
model = build_model() # initialize model

results = {}

# sequential training
for i, (topic, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on topic: {topic} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val])) # fine-tune on train + val

    y_pred = model.predict(X_test)
    print(f"Classification Report after topic {topic}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after topic {topic}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after topic {topic}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all topics
    print("\n--- Evaluation on all topics ---")
    results[topic] = {}
    for test_topic, test_data in datasets.items(): # for each topic
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        f1 = f1_score(y_te, preds, average="weighted")
        results[topic][test_topic] = f1
        print(f"Evaluation on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Phase 1: Training/Fine-tuning on topic: politics ===
Classification Report after topic politics:
              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94     10094
         1.0       0.93      0.93      0.93      9402

    accuracy                           0.93     19496
   macro avg       0.93      0.93      0.93     19496
weighted avg       0.93      0.93      0.93     19496

Confusion Matrix after topic politics:
[[9476  618]
 [ 657 8745]]

Weighted F1-score after topic politics: 0.934597057541472

--- Evaluation on all topics ---
Evaluation on topic politics: Weighted F1 = 0.9346
Evaluation on topic general: Weighted F1 = 0.3693
Evaluation on topic covid: Weighted F1 = 0.3599
Evaluation on topic syria: Weighted F1 = 0.5463
Evaluation on topic islam: Weighted F1 = 0.4834
Evaluation on topic notredame: Weighted F1 = 0.4718
Evaluation on topic gossip: Weighted F1 = 0.3743

=== Phase 2: Training/Fine-tuning on topic: general ===
Classific

In [9]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for topic, res in results.items():
    print(f"\nResults after training on topic {topic}:")
    for test_topic, f1 in res.items():
        print(f"  Test on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on topic politics:
  Test on topic politics: Weighted F1 = 0.9346
  Test on topic general: Weighted F1 = 0.3693
  Test on topic covid: Weighted F1 = 0.3599
  Test on topic syria: Weighted F1 = 0.5463
  Test on topic islam: Weighted F1 = 0.4834
  Test on topic notredame: Weighted F1 = 0.4718
  Test on topic gossip: Weighted F1 = 0.3743

Results after training on topic general:
  Test on topic politics: Weighted F1 = 0.3690
  Test on topic general: Weighted F1 = 0.9655
  Test on topic covid: Weighted F1 = 0.3920
  Test on topic syria: Weighted F1 = 0.3803
  Test on topic islam: Weighted F1 = 0.4487
  Test on topic notredame: Weighted F1 = 0.4654
  Test on topic gossip: Weighted F1 = 0.4172

Results after training on topic covid:
  Test on topic politics: Weighted F1 = 0.4845
  Test on topic general: Weighted F1 = 0.3855
  Test on topic covid: Weighted F1 = 0.9314
  Test on topic syria: Weighted F1 = 0.5101
  Test on topic islam: Weighted F

## VERSION 3: Dataset by Date

In [10]:
dataset_df = data_by_date()

for date, df in dataset_df.items():
    print(f"Date: {date}, Number of samples: {len(df)}")

Date: 2011-2013, Number of samples: 55
Date: 2014, Number of samples: 114
Date: 2015, Number of samples: 84
Date: 2016, Number of samples: 63018
Date: 2017, Number of samples: 16657
Date: 2019, Number of samples: 554
Date: 2020, Number of samples: 10559


In [11]:
# ------------------------------
# Fine-tuning on Dataset by Date
# ------------------------------

dataset_df = data_by_date()

datasets = {date: split_dataset(df) for date, df in dataset_df.items()} # split all datasets in train/val/test
model = build_model() # initialize model

results = {}

# sequential training
for i, (date, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on date: {date} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val])) # fine-tune on train + val

    y_pred = model.predict(X_test)
    print(f"Classification Report after date {date}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after date {date}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after date {date}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all dates
    print("\n--- Evaluation on all dates ---")
    results[date] = {}
    for test_date, test_data in datasets.items(): # for each date
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        f1 = f1_score(y_te, preds, average="weighted")
        results[date][test_date] = f1
        print(f"Evaluation on date {test_date}: Weighted F1 = {f1:.4f}")


=== Phase 1: Training/Fine-tuning on date: 2011-2013 ===
Classification Report after date 2011-2013:
              precision    recall  f1-score   support

         0.0       0.50      0.60      0.55         5
         1.0       0.60      0.50      0.55         6

    accuracy                           0.55        11
   macro avg       0.55      0.55      0.55        11
weighted avg       0.55      0.55      0.55        11

Confusion Matrix after date 2011-2013:
[[3 2]
 [3 3]]

Weighted F1-score after date 2011-2013: 0.5454545454545454

--- Evaluation on all dates ---
Evaluation on date 2011-2013: Weighted F1 = 0.5455
Evaluation on date 2014: Weighted F1 = 0.3843
Evaluation on date 2015: Weighted F1 = 0.5193
Evaluation on date 2016: Weighted F1 = 0.4763
Evaluation on date 2017: Weighted F1 = 0.6145
Evaluation on date 2019: Weighted F1 = 0.3161
Evaluation on date 2020: Weighted F1 = 0.4206

=== Phase 2: Training/Fine-tuning on date: 2014 ===
Classification Report after date 2014:
     

In [12]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for date, res in results.items():
    print(f"\nResults after training on date {date}:")
    for test_date, f1 in res.items():
        print(f"  Test on date {test_date}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on date 2011-2013:
  Test on date 2011-2013: Weighted F1 = 0.5455
  Test on date 2014: Weighted F1 = 0.3843
  Test on date 2015: Weighted F1 = 0.5193
  Test on date 2016: Weighted F1 = 0.4763
  Test on date 2017: Weighted F1 = 0.6145
  Test on date 2019: Weighted F1 = 0.3161
  Test on date 2020: Weighted F1 = 0.4206

Results after training on date 2014:
  Test on date 2011-2013: Weighted F1 = 0.3409
  Test on date 2014: Weighted F1 = 0.4283
  Test on date 2015: Weighted F1 = 0.4669
  Test on date 2016: Weighted F1 = 0.3812
  Test on date 2017: Weighted F1 = 0.2986
  Test on date 2019: Weighted F1 = 0.2984
  Test on date 2020: Weighted F1 = 0.4257

Results after training on date 2015:
  Test on date 2011-2013: Weighted F1 = 0.2424
  Test on date 2014: Weighted F1 = 0.4326
  Test on date 2015: Weighted F1 = 0.4669
  Test on date 2016: Weighted F1 = 0.5046
  Test on date 2017: Weighted F1 = 0.8684
  Test on date 2019: Weighted F1 = 0.6112
 